In [ ]:
import numpy as np
import pandas as pd

from lidar_object_tracking.dataset import load_json_data
from lidar_object_tracking.config import PROCESSED_DATA_DIR
from lidar_object_tracking.plots import plot_with_slider

In [ ]:
dataset = load_json_data(data_directory = PROCESSED_DATA_DIR,scene_num= '004')

In [ ]:
plot_with_slider(dataset)

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps = 0.1, min_samples=10)

clustering_by_frame = {}
for idx in range(80):
    clustering_by_frame[idx] = db.fit(dataset[idx])

In [ ]:
for idx in range(80):
    dataset[idx]['label'] = clustering_by_frame[idx].labels_

In [ ]:
dataset[0][dataset[0]['label'] == 0]
        
        

In [ ]:
frame = dataset[0].groupby('label')[['xr','yr','z']].mean()

frame.loc[2]

In [ ]:
center_of_mass_dict = {}

for idx in range(len(dataset)):
    centroid_by_label = {}
    cluster_centers = dataset[idx].groupby('label')[['xr','yr','z']].mean()
    for label in cluster_centers.index:
        if label >-1:
            centroid_by_label[label] = list(cluster_centers.loc[label][['xr','yr','z']])
    center_of_mass_dict[idx] = centroid_by_label

center_of_mass_dict
    

In [ ]:

def euclid_dist(vector1, vector2):
    num_coord = len(vector1)
    sum_squares = np.sum([(vector1[coord] - vector2[coord])**2 for coord in range(num_coord)])
    return sum_squares**.5

vector1 = [3,0,1]
vector2 = [0,4,3]

euclid_dist(vector1,vector2)

In [ ]:
vector1 = center_of_mass_dict[0][0]

distances = np.zeros(len(center_of_mass_dict[1]))
print(distances)
for key in center_of_mass_dict[1].keys():
    dist = euclid_dist(vector1, center_of_mass_dict[1][key])
    distances[key] = dist

print(np.argmin(distances))


In [ ]:
track_clusters = {}
for key in center_of_mass_dict[0].keys():
    vector1 = center_of_mass_dict[0][key]
    track_cluster_by_frame = [key]
    for idx in range(1,len(center_of_mass_dict)):
        distances = np.zeros(len(center_of_mass_dict[idx]))
        for label in center_of_mass_dict[idx].keys():
            dist = euclid_dist(vector1, center_of_mass_dict[idx][label])
            distances[label] = dist
        next_label = int(np.argmin(distances))
        vector1 = center_of_mass_dict[idx][next_label]
        track_cluster_by_frame.append(next_label)

    track_clusters[track_cluster_by_frame[0]] = track_cluster_by_frame

In [ ]:
track_clusters

In [ ]:
dataset[0][dataset[0]['label'] == 1]

In [ ]:
dataset0 = {}
cluster_id = 0

cluster_tracker = track_clusters[cluster_id]

for i in range(len(dataset)):
    print(cluster_tracker[i])
    dataset0[i] = dataset[i][dataset[i]['label'] == cluster_tracker[i]]
    print(dataset0[i])

In [ ]:
plot_with_slider(dataset0)